# NutriChat — RAG Pipeline Notebook


In [1]:
import os
from getpass import getpass

from pathlib import Path
from docling.document_converter import DocumentConverter
import pandas as pd

from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter

import chromadb
from chromadb.utils import embedding_functions

import ollama

import json
import shutil

# Set Hugging Face Token to avoid rate limits and warnings
hf_token = getpass("Enter your Hugging Face Token: ")
os.environ["HF_TOKEN"] = hf_token
print("HF_TOKEN set successfully!")

HF_TOKEN set successfully!


## 2.1 Load & Inspect


### Load & Inspect Analysis
- **Number of Documents**: 3
- **Total Pages**: 201
- **Formats**: PDF (.pdf)
- **Parsing Results**: All files parsed successfully. No files failed.
- **OCR Notes**: Docling automatically applied OCR where needed; some pages returned empty OCR results, but the overall document structure was recovered.

In [2]:
# Initialize the Docling converter
converter = DocumentConverter()

DATA_DIR = Path("../Data") 
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))

print(f"Found {len(pdf_paths)} PDF files(s):")
for p in pdf_paths:
    print(f" - {p.name} ({p.stat().st_size / 1_000_000:.1f} MB)")

Preset 'granite_vision_v4' already registered for ChartExtractionVlmEngineOptions


Found 3 PDF files(s):
 - 9789240101876-eng.pdf (1.2 MB)
 - Dietary_Guidelines_for_Americans_2020-2025.pdf (32.2 MB)
 - WHO_healthy_diet_factsheet.pdf (0.1 MB)


In [3]:
# Convert PDFs to a structured format (Markdown)
converted_docs = []
failures = []

for path in pdf_paths:
    print(f"Converting {path.name}...")
    try:
        result = converter.convert(path)
        markdown_text = result.document.export_to_markdown()
        
        # Docling documents have a .pages attribute
        num_pages = len(result.document.pages)
        
        converted_docs.append({
            "source": path.name,
            "pages": num_pages,
            "text": markdown_text,
            "format": path.suffix
        })
    except Exception as e:
        print(f"Error converting {path.name}: {e}")
        failures.append({"source": path.name, "error": str(e)})

# Convert to DataFrame
df_docs = pd.DataFrame(converted_docs)
df_failures = pd.DataFrame(failures)

# Calculate summary for the markdown cell
total_docs = len(df_docs)
total_pages = df_docs['pages'].sum() if not df_docs.empty else 0
formats = df_docs['format'].unique().tolist() if not df_docs.empty else []
failed_files = df_failures['source'].tolist() if not df_failures.empty else []

print("\n--- Dataset Summary ---")
print(f"Total Documents: {total_docs}")
print(f"Total Pages: {total_pages}")
print(f"Formats: {formats}")
print(f"Failed Files: {failed_files}")
print("-----------------------\n")

df_docs[['source', 'pages', 'format']].head()

Converting 9789240101876-eng.pdf...


[INFO] 2026-09-18 21:02:58,955 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-18 21:02:58,964 [RapidOCR] download_file.py:60: File exists and is valid: /Users/ahmed/NutriChat/.venv/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-18 21:02:58,965 [RapidOCR] main.py:63: Using /Users/ahmed/NutriChat/.venv/lib/python3.13/site-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-09-18 21:02:59,012 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-18 21:02:59,014 [RapidOCR] download_file.py:60: File exists and is valid: /Users/ahmed/NutriChat/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-18 21:02:59,015 [RapidOCR] main.py:63: Using /Users/ahmed/NutriChat/.venv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-18 21:02:59,031 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-18 21:02:59,04

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-09-18 21:03:05,982 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-09-18 21:03:07,123 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-09-18 21:03:07,979 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!


2026-09-18 21:03:20,629 MatchingPostProcessor WARNING  Orphan pdf_cell 155 recovered to col=2 by nearest-column fallback (row=3, x=797.6, dist=157.4)
2026-09-18 21:03:20,630 MatchingPostProcessor WARNING  Orphan pdf_cell 156 recovered to col=3 by nearest-column fallback (row=3, x=835.1, dist=138.5)


...

RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-09-18 21:04:45,927 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-09-18 21:05:08,823 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
RapidOCR returned empty result!
[WARNING] 2026-09-18 21:06:06,607 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-09-18 21:06:41,393 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
[WARNING] 2026-09-18 21:06:42,188 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!
RapidOCR returned empty

2026-09-18 21:11:37,360 MatchingPostProcessor WARNING  Orphan pdf_cell 36 recovered to row=2 by nearest-row fallback (col=6, y=268.1, dist=29.7)


[WARNING] 2026-09-18 21:12:26,058 [RapidOCR] main.py:132: The text detection result is empty
RapidOCR returned empty result!


Converting WHO_healthy_diet_factsheet.pdf...

--- Dataset Summary ---
Total Documents: 3
Total Pages: 201
Formats: ['.pdf']
Failed Files: []
-----------------------



,source,pages,format
0,9789240101876-eng.pdf,23,.pdf
1,Dietary_Guidelines_for_Americans_2020-2025.pdf,164,.pdf
2,WHO_healthy_diet_factsheet.pdf,14,.pdf


## 2.2 Chunking Strategy

### Strategy Justification
For NutriChat, I have implemented a **Hybrid Markdown-Aware Strategy**:

1.  **Semantic Splitting (Headers)**: I use a `MarkdownHeaderTextSplitter` to first split the document by its structure (`#`, `##`, `###`). This ensures that a specific nutrition topic (e.g., "Vitamin D Guidelines") is treated as a distinct semantic unit.
2.  **Recursive Character Splitting**: Since some sections are still too large for an LLM's context window, I apply a `RecursiveCharacterTextSplitter` to those sections.
3.  **Chunk Size (1000 chars)**: Chosen because nutrition facts are often dense. 1000 characters provide enough context to capture a full recommendation or a small table without introducing too much noise.
4.  **Overlap (100 chars)**: A 10% overlap is used to ensure that if a critical piece of information (like a dosage or limit) is split across two chunks, the context is preserved in both.

In [4]:
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100
)

all_chunks = []

for idx, row in df_docs.iterrows():
    # Split by Header
    header_splits = markdown_splitter.split_text(row['text'])
    
    # split large sections into smaller chunks
    for hs in header_splits:
        final_chunks = text_splitter.split_text(hs.page_content)
        for chunk in final_chunks:
            meta = hs.metadata.copy()
            meta["source"] = row['source']

            all_chunks.append({
                "doc_id": idx,
                "source": row['source'],
                "text": chunk,
                "metadata": meta
            })

# Create a DataFrame of chunks for the Vector DB step
df_chunks = pd.DataFrame(all_chunks)

print(f"Original documents: {len(df_docs)}")
print(f"Total chunks created: {len(df_chunks)}")
df_chunks.head()

Original documents: 3
Total chunks created: 1079


,doc_id,source,text,metadata
0,0,9789240101876-eng.pdf,Joint statement by the Food and Agriculture Or...,"{'Header 2': 'What are healthy diets?', 'sourc..."
1,0,9789240101876-eng.pdf,Some rights reserved. This work is available u...,{'Header 2': '© World Health Organization and ...
2,0,9789240101876-eng.pdf,"Under the terms of this licence, you may copy,...",{'Header 2': '© World Health Organization and ...
3,0,9789240101876-eng.pdf,Any mediation relating to disputes arisin...,{'Header 2': '© World Health Organization and ...
4,0,9789240101876-eng.pdf,Third-party materials. If you wish to reuse ma...,{'Header 2': '© World Health Organization and ...


## 2.3 Embeddings & Vector Store

**Chosen Components:**
- **Embedding Model**: `sentence-transformers/all-MiniLM-L6-v2`. This model is lightweight and highly efficient for mapping nutrition-related text to a 384-dimensional vector space.
- **Vector Store**: `ChromaDB`. I chose Chroma because it provides native persistence to disk, allows for easy metadata filtering, and is specifically designed for RAG pipelines.
- **Persistence**: The store is persisted to the `./vector_db` directory, allowing the backend to load the pre-computed index without needing to re-process the PDFs.

In [5]:
# 1. Initialize the Embedding Function
embedding_func = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# 2. Initialize ChromaDB Client with Persistence
client = chromadb.PersistentClient(path="../vector_db")

# 3. Create or Get the Collection
collection = client.get_or_create_collection(
    name="nutrichat_knowledge", 
    embedding_function=embedding_func
)

# 4. Prepare data for indexing
# Chroma expects IDs, Documents (text), and Metadatas
ids = [str(i) for i in range(len(df_chunks))]
documents = df_chunks['text'].tolist()
metadatas = df_chunks['metadata'].tolist()

# 5. Index the chunks into the Vector Store
batch_size = 100
for i in range(0, len(documents), batch_size):
    collection.add(
        ids=ids[i : i + batch_size],
        documents=documents[i : i + batch_size],
        metadatas=metadatas[i : i + batch_size]
    )

print(f"Indexing complete!")
print(f"Stored {len(documents)} chunks in ChromaDB at './vector_db'")

# Test a quick query to verify it works
test_query = "What is a healthy diet?"
results = collection.query(
    query_texts=[test_query],
    n_results=2
)

print("\n--- Test Query Result ---")
print(f"Query: {test_query}")
for doc in results['documents'][0]:
    print(f"Result: {doc[:200]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexing complete!
Stored 1079 chunks in ChromaDB at './vector_db'

--- Test Query Result ---
Query: What is a healthy diet?
Result: - Diet plays a critical role in shaping the health and well-being of both individuals and populations, and unhealthy diets are a major risk factor for disease and disability.
- Healthy diets help to p...
Result: Consuming a healthy diet throughout the lifecourse helps to prevent malnutrition in all its forms as well as a range of noncommunicable diseases (NCDs) and conditions.  
However, change in food produc...


## 2.4 Retrieval & Prompting

**Approach:**
- Retrieve the top-k most relevant chunks from ChromaDB for a given question (k=4, balancing enough context vs. noise/context-window limits for a 3B local model).
- Build a prompt that explicitly instructs the model to answer *only* from the retrieved context, cite which source document each fact came from, and say it doesn't know rather than guessing if the context doesn't contain the answer.
- Generate with Ollama running `llama3.2` locally — no external API calls, everything runs on-device.

In [ ]:
LLM_MODEL = "llama3.2"
TOP_K = 4

def retrieve(query: str, top_k: int = TOP_K):
    """Query the Chroma collection and return chunks + metadata + distances."""
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )
    chunks = results['documents'][0]
    metadatas = results['metadatas'][0]
    distances = results['distances'][0]
    return list(zip(chunks, metadatas, distances))  

def build_prompt(query: str, retrieved_chunks) -> str:
    """Assemble a context block with source labels + the instruction prompt."""
    context_blocks = []
    for i, (chunk, metadata, distance) in enumerate(retrieved_chunks):
        source = metadata.get("source", "Unknown Source")
        context_blocks.append(f"[Source {i}: {source}]\n{chunk}")
    context_text = "\n\n".join(context_blocks)

    prompt = f"""You are NutriChat, a nutrition assistant. Answer the question using ONLY the context below.

        Rules:
        - If the context does not contain the answer, say "I don't have enough information in my sources to answer that." Do not guess or use outside knowledge.
        - When you use a fact, cite it with the source label it came from, e.g. (Source 1).
        - Be concise and direct.

        Context:
        {context_text}

        Question: {query}

        Answer:"""
    return prompt

def rag_query (query: str, top_k: int = TOP_K, verbose: bool = False) -> str:
    """Full RAG call: retrieve -> prompt -> generate. Returns answer + source list."""
    retrieved = retrieve(query, top_k)
    prompt = build_prompt(query, retrieved)

    if verbose:
        print(prompt)
        print("-" * 80)

    response = ollama.chat(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = response['message']['content']
    sources = sorted({meta.get("source", "unknown") for _c, meta, _d in retrieved})
    return {"question": query, "answer": answer, "sources": sources, "retrieved": retrieved}



In [7]:
for q in [
    "How many cups of vegetables should an adult eat per day?",
    "What is a healthy diet according to WHO?",
    "What is the capital of France?",  # deliberately off-topic — should trigger the fallback, not a hallucination
]:
    result = rag_query(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer']}")
    print(f"Sources: {result['sources']}")
    print("=" * 80)


Q: How many cups of vegetables should an adult eat per day?
A: According to Source 2 (Dietary_Guidelines_for_Americans_2020-2025.pdf), an adult should eat ⅔ cup of vegetables per day.
Sources: ['9789240101876-eng.pdf', 'Dietary_Guidelines_for_Americans_2020-2025.pdf', 'WHO_healthy_diet_factsheet.pdf']
Q: What is a healthy diet according to WHO?
A: According to the World Health Organization (Source 1 and Source 2), a healthy diet is characterized by four core principles: 

1. Adequacy: meets, without exceeding, micronutrient and macronutrient needs such that deficiencies are prevented.
2. Balance: 
3. Moderation: 
4. Diversity: 

Additionally, a healthy diet must also be safe, meaning it is free from microbial and chemical contaminants (Source 2).
Sources: ['9789240101876-eng.pdf', 'WHO_healthy_diet_factsheet.pdf']
Q: What is the capital of France?
A: I don't have enough information in my sources to answer that.
Sources: ['Dietary_Guidelines_for_Americans_2020-2025.pdf']


## 2.5 (Skipped — Core Track)

## 2.6 Evaluation

10+ test questions covering the corpus, plus one deliberately off-topic control question to check the system says "I don't know" rather than hallucinating.

In [8]:
eval_questions = [
    "How many cups of vegetables should an adult eat per day?",
    "What is the recommended daily limit for added sugars?",
    "How much sodium is recommended per day for adults?",
    "What dietary advice is given for women during pregnancy?",
    "What defines a healthy diet according to the WHO/FAO?",
    "What are the health risks of an unhealthy diet?",
    "What nutrients are considered of public health concern in the US?",
    "What dietary recommendations exist for older adults?",
    "What is the recommended limit for saturated fat intake?",
    "How much added sugar should children consume?",
    "What role does physical activity play alongside a healthy diet?",
    "What is the capital of France?",  # off-topic control — expect a refusal, not a hallucinated answer
]

eval_records = []
for q in eval_questions:
    result = rag_query(q)
    eval_records.append({
        "question": result["question"],
        "answer": result["answer"],
        "sources": ", ".join(result["sources"]),
    })

eval_df = pd.DataFrame(eval_records)
pd.set_option('display.max_colwidth', None)
eval_df


,question,answer,sources
0,How many cups of vegetables should an adult eat per day?,"According to Source 2, an adult should eat ⅔ cup of vegetables per day (Source 2).","9789240101876-eng.pdf, Dietary_Guidelines_for_Americans_2020-2025.pdf, WHO_healthy_diet_factsheet.pdf"
1,What is the recommended daily limit for added sugars?,"According to the Dietary Guidelines for Americans 2020-2025 (Source 3) and the World Health Organization (Source 1 and 2), the recommended daily limit for added sugars is less than 10 percent of total daily energy intake.","9789240101876-eng.pdf, Dietary_Guidelines_for_Americans_2020-2025.pdf, WHO_healthy_diet_factsheet.pdf"
2,How much sodium is recommended per day for adults?,"According to Source 1, sodium intake should be restricted to 2 grams per day (corresponding to 5 grams of table salt, i.e., sodium chloride) in adults.","9789240101876-eng.pdf, Dietary_Guidelines_for_Americans_2020-2025.pdf"
3,What dietary advice is given for women during pregnancy?,"According to the Dietary Guidelines for Americans 2020-2025 (Source 1, Source 2, and Source 3), women during pregnancy are advised to follow a healthy dietary pattern, which includes:\n\n1. Taking a daily prenatal vitamin and mineral supplement in addition to consuming a healthy dietary pattern (Source 1).\n2. Eating nutrient-dense food choices to meet increased calorie and nutrient needs (Source 3).\n3. Following the Healthy U.S.-Style Dietary Pattern, which includes specific amounts and limits for food groups and other dietary components (Source 2).\n4. Avoiding foods high in added sugars, saturated fat, and sodium (Source 3).\n\nThese guidelines aim to help women meet the nutritional needs of pregnancy and ensure a healthy weight status, short-term and long-term health benefits for the mother and her child.",Dietary_Guidelines_for_Americans_2020-2025.pdf
4,What defines a healthy diet according to the WHO/FAO?,"According to the WHO/FAO, a healthy diet is defined by four core principles: adequacy, balance, moderation, and diversity. (Source 3)","9789240101876-eng.pdf, WHO_healthy_diet_factsheet.pdf"
5,What are the health risks of an unhealthy diet?,"According to Source 1 (Source 1: WHO_healthy_diet_factsheet.pdf) and Source 3 (Source 3: WHO_healthy_diet_factsheet.pdf), an unhealthy diet is associated with negative health outcomes, including:\n\n- Increased risk of noncommunicable diseases (NCDs), such as diabetes, heart disease, stroke, and cancer.\n- Risk of malnutrition in all its forms.\n- Increased risk of diet-related noncommunicable diseases.\n- Negative health outcomes associated with consuming foods high in unhealthy fats, free sugars, and sodium.\n\n(Source 2: WHO_healthy_diet_factsheet.pdf) also mentions that diets containing significant amounts of highly processed foods are associated with negative health outcomes.","Dietary_Guidelines_for_Americans_2020-2025.pdf, WHO_healthy_diet_factsheet.pdf"
6,What nutrients are considered of public health concern in the US?,"According to the Dietary Guidelines for Americans 2020-2025, calcium, potassium, dietary fiber, and vitamin D are considered dietary components of public health concern for the general US population because low intakes are associated with health concerns.",Dietary_Guidelines_for_Americans_2020-2025.pdf
7,What dietary recommendations exist for older adults?,"According to the Dietary Guidelines for Americans 2020-2025, older adults (ages 60 and older) should:\n\n* Limit foods and beverages higher in added sugars, saturated fat, and sodium, and limit alcoholic beverages.\n* Increase consumption of:\n + Fruit\n + Vegetables\n + Whole grains\n + Dairy\n* Ensure protein intake meets recommendations.\n* Choose nutrient-dense options within each food group.\n* Consume appropriate portion sizes because calorie needs decline with age.\n\n(This information comes from Sources 2 and 3.)",Dietary_Guidelines_for_Americans_2020-2025.pdf
8,What is the recommended limit for saturated fat intake?,"

In [9]:
eval_df.to_csv("../eval_results.csv", index=False)
print("Saved to ../eval_results.csv — open it, fill in human_judgment + notes for each row, then re-save.")


Saved to ../eval_results.csv — open it, fill in human_judgment + notes for each row, then re-save.


**Summary:** 9/12 grounded, 3/12 partially grounded, 0/12 fully hallucinated. The off-topic control question was handled correctly, confirming the prompt's refusal instruction works rather than the model defaulting to outside knowledge.

**Main failure cases observed:**

None of the answers fabricated information outright, but three showed a subtler failure mode: retrieving a technically-relevant chunk and stating it confidently, without checking that it actually answered the question at the right level of specificity. The vegetables question (#1) pulled what looks like a specific subcategory number and presented it as the general adult recommendation. The sodium question (#3) got the right numbers but attributed them to the wrong source document — likely because two of our source PDFs (USDA and WHO) both discuss sodium limits with similar-sounding numbers, and the model's citation didn't track which chunk the number actually came from. The children's sugar question (#10) answered "how much do children currently consume" instead of "how much should they consume" — a sign retrieval matched on the sugar/children keywords without distinguishing descriptive statistics from prescriptive guidance in the chunk.

**Mitigations applied / considered:**

- Kept the off-topic control question in the eval set specifically to verify the refusal path works — it did, with zero hallucination on a completely unrelated question.
- The prompt's citation requirement made these errors visible and checkable at all — without per-source labels in the answer, the sodium misattribution (#3) would have been invisible.
- For future improvement: increasing `TOP_K` from 4 slightly, or adding a "state which document this specific number came from, and don't average or generalize across documents" instruction, would likely reduce the cross-source mixing seen in #1 and #3. For #10, rephrasing the prompt to explicitly distinguish "current intake" language from "recommended limit" language in retrieved chunks would help avoid answering the wrong version of a similar question.

## 2.7 Export




In [10]:
# 1. Define the configuration
config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "chunk_size": 1000,
    "chunk_overlap": 100,
    "collection_name": "nutrichat_knowledge",
    "vector_db_path": "../vector_db"
}

# 2. Create the export directory
export_dir = Path("../export")
export_dir.mkdir(exist_ok=True)

# 3. Save the config as JSON
config_path = export_dir / "config.json"
with open(config_path, "w") as f:
    json.dump(config, f, indent=4)

# 4. Export the vector store
# We copy the persistent directory to the export folder
db_source = Path("../vector_db")
db_dest = export_dir / "vector_db"

if db_source.exists():
    if db_dest.exists():
        shutil.rmtree(db_dest)
    shutil.copytree(db_source, db_dest)

print(f"Export complete!")
print(f"Config saved to: {config_path}")
print(f"Vector store exported to: {db_dest}")
print("\nYour backend can now load the 'export' folder directly.")

Export complete!
Config saved to: ../export/config.json
Vector store exported to: ../export/vector_db

Your backend can now load the 'export' folder directly.
